# 🏥 AH Ameliyat Vaka Hacmi Tahmini — v1 (2020–2026)
### LSTM · Facebook Prophet · LightGBM · Weighted Ensemble
**Kaynak:** Bui et al. (2023) — *IISE Transactions on Healthcare Systems Engineering*

---

## 📊 Veri Seti Profili
| Özellik | Değer |
|---|---|
| **Ameliyat Verisi** | AH_2020_2026_Ameliyat.xlsx (93.067 kayıt) |
| **Doluluk Verisi** | AH_Hastane_Doluluk_Oranı.xlsx (Ara 2022 – Kas 2025, günlük) |
| **Eğitim** | Oca 2020 – Ara 2024 · **Validasyon** Oca 2025 – Ara 2025 |
| **Tahmin** | Oca 2026 – Mar 2026 (gerçek veri mevcut — model karşılaştırması yapılabilir) |

## ℹ️ KUH Notebook'undan Farklar
| # | Alan | Değişiklik |
|---|---|---|
| 1 | **Veri** | AH dosyaları — 93.067 kayıt, 2026-04-13'e kadar gerçek veri |
| 2 | **Tahmin dönemi** | 2026 Oca-Mar için gerçek veri mevcut → metrik hesabı aktif |
| 3 | **Doluluk sütunları** | 'Tarih'/'Doluluk' (KUH'ta farklı format) — otomatik işlenir |
| 4 | **Çıktı dosyaları** | ah_* öneki ile kaydedilir |

---
### ▶️ Kullanım
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Her iki Excel dosyasını sırayla yükle


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Bağımlılıkları Kur                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Google Colab ortamına varsayılan olarak yüklü gelmeyen üç paketi kurar:
#   • prophet  — Facebook'un zaman serisi tahmin kütüphanesi
#   • openpyxl — Excel (.xlsx) dosyalarını okumak için
#   • lightgbm — Hızlı ve bellek açısından verimli Gradient Boosting ağaçları
#
# Değiştirilecek bir şey yok. Lokal ortamda çalıştırıyorsan bu hücreyi
# atlayabilirsin (paketlerin zaten kuruluysa).
#
!pip install prophet openpyxl lightgbm -q
print('✔ Kurulum tamamlandı (prophet · openpyxl · lightgbm)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Import & Merkezi Konfigürasyon Paneli                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Tüm kütüphaneleri tek seferde içe aktarır ve projedeki tüm parametreleri
#   tek bir yerde toplar. Diğer hücreler bu değişkenleri doğrudan referans
#   alır — parametre denemelerinde sadece BU hücreyi düzenleyip 'Run All'
#   yapmak yeterlidir.
#
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, RepeatVector,
    TimeDistributed, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from prophet import Prophet
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import lightgbm as lgb

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  TUNING PANELİ — tüm parametreler tek yerde
# ─────────────────────────────────────────────────────────────────────────────

# ── LSTM — Mimari ─────────────────────────────────────────────────────────────
SEED              = 42

LOOKBACK          = 28
# [TEST ET] Geçmişe bakış penceresi (gün). Model bu kadar günü 'görerek' tahmin yapar.
# Dene: 21 → 28 → 42 → 56
# Öneri: 42 veya 56 dene — 2 aylık hafıza, mevsimsel örüntüler için daha güçlü bağlam.

HORIZON           = 7
# [SABİT] Kaç gün ileriye tahmin yapılacağı. 7 = haftalık blok.
# Değiştirirsen direct_forecast_lgb ve make_seq_flat fonksiyonları da güncellenmeli.

LSTM_UNITS        = 64
# [TEST ET] Encoder ve decoder LSTM katmanlarının genişliği (gizli birim sayısı).
# Dene: 64 → 96 → 128
# AH verisi KUH'tan ~%36 daha büyük (93k kayıt) — 96 veya 128 daha zengin temsil öğrenebilir.

DROPOUT           = 0.3
# [TEST ET] İleri besleme katmanlarındaki dropout oranı.
# Dene: 0.2 → 0.3 → 0.4
# Val kaybı eğitim kaybından çok yüksekse (overfitting) 0.4'e çık.

RECURRENT_DROPOUT = 0.1
# [TEST ET] LSTM gizli durum bağlantılarındaki dropout.
# Dene: 0.0 → 0.1 → 0.2
# GPU kullanıyorsan 0.0 daha hızlı (cuDNN optimizasyonu aktif kalır).

HUBER_DELTA       = 1.0
# [TEST ET] Huber kayıp fonksiyonunun eşik değeri.
# Dene: 0.5 → 1.0 → 2.0
# AH günlük max=92 vaka ile KUH'a benzer aralık — 1.0 iyi başlangıç noktası.

EPOCHS            = 200
# EarlyStopping aktif olduğundan bu değer maksimum; genellikle çok önce durur.

BATCH_SIZE        = 16
# [TEST ET] Her gradient adımında işlenen dizi sayısı.
# Dene: 16 → 32
# 32 daha düzgün gradyan verir; AH verisi büyük olduğundan 32 tercih edilebilir.

LEARNING_RATE     = 3e-4
# [OPTİMUM] ReduceLROnPlateau dinamik olarak daha da düşüreceği için dengeli başlangıç.

CLIP_NORM         = 5.0
# [OPTİMUM] Gradient norm kırpma. Exploding gradient görülürse 1.0'a düşür.

EARLY_PATIENCE    = 30
# [OPTİMUM] Val kaybı bu kadar epoch iyileşmezse eğitim durur.
# Dene: 25 → 30 → 40

REDUCE_FACTOR     = 0.5
# [TEST ET] ReduceLROnPlateau katsayısı. Dene: 0.3 → 0.5

REDUCE_PATIENCE   = 15
# [OPTİMUM] LR düşürme için plato toleransı. Dene: 10 → 15 → 20

VAL_SPLIT         = 0.15
# [TEST ET] LSTM iç validasyonu için eğitim setinden ayrılan oran.
# Dene: 0.10 → 0.15 → 0.20

# ── Özellik mühendisliği ───────────────────────────────────────────────────────
COVID_END         = '2022-06-01'
# [TEST ET] Pandemi 'bitti' bayrağının sıfırlandığı tarih.
# Dene: '2022-03-01' → '2022-06-01' → '2022-09-01'
# AH'ın kendi veri görselinden normalleşme tarihini belirle.

HOL_WINDOW        = 3
# [OPTİMUM] Tatil gününden ±kaç günlük yakınlık bayrağı üretilsin.
# Dene: 2 → 3 → 4

DOL_EWMA_SPAN     = 7
# [TEST ET] Doluluk oranı için EWMA penceresi.
# Dene: 3 → 7 → 14 → 30

# ── LightGBM ──────────────────────────────────────────────────────────────────
LGB_N_ESTIMATORS  = 300
# [TEST ET] Toplam ağaç sayısı. Dene: 200 → 300 → 500
# Early stopping eklenirse (Cell 11'e bak) 500-1000'e güvenle çıkabilirsin.

LGB_MAX_DEPTH     = 6
# [TEST ET] Her ağacın maksimum derinliği. Dene: 4 → 6 → 8

LGB_LR            = 0.05
# [OPTİMUM] N_ESTIMATORS ile birlikte ayarla (düşük LR + çok ağaç = daha iyi).

LGB_SUBSAMPLE     = 0.8
# [OPTİMUM] Satır örnekleme oranı. Dene: 0.6 → 0.8 → 0.9

LGB_MIN_LEAF      = 5
# [TEST ET] Yaprak düğümünde minimum örnek. Dene: 3 → 5 → 10

LGB_COLSAMPLE     = 0.8
# [OPTİMUM] Sütun örnekleme oranı. Dene: 0.6 → 0.8 → 1.0

# ── Prophet ───────────────────────────────────────────────────────────────────
PROPHET_CP_SCALE  = 0.15
# [TEST ET] EN KRİTİK PROPHET PARAMETRESİ — trend esnekliği.
# Dene: 0.05 → 0.15 → 0.3 → 0.5
# Bileşen grafiğinde trend düz/garip görünüyorsa bu değeri ayarla.

PROPHET_SEAS_MODE = 'multiplicative'
# [OPTİMUM] Oransal mevsimsel etki. Ameliyat hacmi oransal değişim gösteriyorsa doğru seçim.

PROPHET_SEAS_PRIOR= 10.0
# [TEST ET] Mevsimsellik esneklik sınırı. Dene: 5.0 → 10.0 → 20.0

PROPHET_HOL_PRIOR = 15.0
# [OPTİMUM] Tatil etkisi büyüklük sınırı. Tatillerde sıfıra yakın düşüyorsa 20.0 dene.

PROPHET_YEARLY_N  = 15
# [TEST ET] Yıllık mevsimsellik Fourier bileşeni sayısı. Dene: 10 → 15 → 20

# ── Dönem sınırları ───────────────────────────────────────────────────────────
TRAIN_END      = '2025-01-01'   # Eğitim: 2020-2024
VAL_END        = '2026-01-01'   # Validasyon: 2025
FORECAST_START = '2026-01-01'   # Tahmin başlangıcı
FORECAST_END   = '2026-03-31'   # Tahmin bitişi
# ℹ️  AH verisi 2026-04-13'e kadar mevcut — Oca-Mar 2026 için gerçek veri
#    test_comp'ta dolu olacak → has_actual=True → Cell 13'te gerçek metrikler görünür.

print('✔ Konfigürasyon yüklendi')
print(f'  LOOKBACK={LOOKBACK}  LSTM_UNITS={LSTM_UNITS}  DROPOUT={DROPOUT}')
print(f'  LR={LEARNING_RATE}  clipnorm={CLIP_NORM}  Huber_δ={HUBER_DELTA}')
print(f'  LGB: trees={LGB_N_ESTIMATORS}  depth={LGB_MAX_DEPTH}  lr={LGB_LR}')
print(f'  Prophet: cp_scale={PROPHET_CP_SCALE}  mode={PROPHET_SEAS_MODE}  yearly_N={PROPHET_YEARLY_N}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Dosya Yükleme (Google Colab)                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Google Colab'ın dosya yükleme arayüzünü açar ve iki Excel dosyasını
#   sırayla belleğe alır:
#     1. AH_2020_2026_Ameliyat.xlsx  — 93.067 ameliyat kaydı (2020-01-02 → 2026-04-13)
#     2. AH_Hastane_Doluluk_Oranı.xlsx — Ara 2022-Kas 2025 günlük doluluk oranı
#
# Lokal ortama taşıma:
#   Bu hücreyi kaldırıp aşağıdaki iki satırı Cell 2'nin sonuna ekle:
#     DATA_PATH    = '/tam/yol/AH_2020_2026_Ameliyat.xlsx'
#     DOLULUK_PATH = '/tam/yol/AH_Hastane_Doluluk_Oranı.xlsx'
#
from google.colab import files

print('📂 Lütfen yükleyin: AH_2020_2026_Ameliyat.xlsx')
uploaded1  = files.upload()
DATA_PATH  = list(uploaded1.keys())[0]
print(f'✔ Yüklendi: {DATA_PATH}')

print()
print('📂 Lütfen yükleyin: AH_Hastane_Doluluk_Oranı.xlsx')
uploaded2    = files.upload()
DOLULUK_PATH = list(uploaded2.keys())[0]
print(f'✔ Yüklendi: {DOLULUK_PATH}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Veri Hazırlama & Doluluk Entegrasyonu                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar — üç aşama:
#
# 4a — Ameliyat verisi (AH_2020_2026_Ameliyat.xlsx):
#   'Hizmet Kayıt Tarihi' sütunu (iloc[:,2]) tarih olarak parse edilir →
#   groupby(date).size() ile her gün için toplam vaka sayısı (volume) üretilir →
#   eksik günler fill_value=0 ile tam takvim serisine dönüştürülür.
#   AH veri aralığı: 2020-01-02 → 2026-04-13 (2026 Oca-Mar gerçek veri dahil).
#
# 4b — Doluluk verisi (AH_Hastane_Doluluk_Oranı.xlsx):
#   Dosyada 'Tarih'/'Doluluk' sütunları var — aşağıda ['date','doluluk'] olarak
#   yeniden adlandırılır (otomatik uyum, değiştirme gerekmez).
#
# 4c — Mevsimsel impütasyon:
#   Doluluk verisi Ara 2022'de başlıyor; 2020-2022 arası ve 2026 için
#   aynı takvim ayının gerçek ortalaması kullanılır.
#   ✅ DÜZELTİLDİ: dol_full artık daily'nin gerçek bitiş tarihine kadar uzanıyor.
#   Sabit '2026-03-31' yerine daily['date'].max() kullanılıyor —
#   böylece daily içindeki hiçbir gün NaN doluluk değeri almaz.
#
# ⚠️  DİKKAT: df_raw.iloc[:, 2] tarih sütununun Excel'de 3. kolonda (C sütunu,
#   'Hizmet Kayıt Tarihi') olduğunu varsayar — AH dosyasında doğrulanmıştır.
#
# ── 4a: Ameliyat verisi ──────────────────────────────────────────────────────
df_raw = pd.read_excel(DATA_PATH)
df_raw['date'] = pd.to_datetime(df_raw.iloc[:, 2], errors='coerce').dt.normalize()
df_raw = df_raw.dropna(subset=['date'])

daily_raw  = df_raw.groupby('date').size().reset_index(name='volume')
full_range = pd.date_range(daily_raw['date'].min(), daily_raw['date'].max(), freq='D')
daily = (daily_raw.set_index('date')
                  .reindex(full_range, fill_value=0)
                  .reset_index()
                  .rename(columns={'index': 'date'}))

df_raw['month_key'] = df_raw['date'].dt.to_period('M')
prov_monthly = (df_raw.groupby('month_key')['Primer Doktor Kodu']
                      .nunique().reset_index(name='num_prov'))

print(f'✔ Ameliyat: {len(df_raw):,} kayıt yüklendi')
print(f'  Tarih aralığı : {daily["date"].min().date()} → {daily["date"].max().date()}')
print(f'  Toplam gün    : {len(daily)}  (veri olan: {(daily.volume > 0).sum()})')
print(f'  Günlük hacim  : min={daily.volume.min()}  maks={daily.volume.max()}  ort={daily.volume.mean():.1f}')

# ── 4b: Doluluk verisi yükle ─────────────────────────────────────────────────
df_dol = pd.read_excel(DOLULUK_PATH)
df_dol.columns = ['date', 'doluluk']   # AH formatı: Tarih/Doluluk → date/doluluk
df_dol['date']    = pd.to_datetime(df_dol['date']).dt.normalize()
df_dol['doluluk'] = pd.to_numeric(df_dol['doluluk'], errors='coerce')
df_dol = df_dol.dropna().drop_duplicates('date').set_index('date').sort_index()

print(f'\n✔ Doluluk: {len(df_dol):,} gün yüklendi')
print(f'  Tarih aralığı : {df_dol.index.min().date()} → {df_dol.index.max().date()}')
print(f'  Aralık        : {df_dol.doluluk.min():.2f} – {df_dol.doluluk.max():.2f}  (ort={df_dol.doluluk.mean():.3f})')

# ── 4c: Mevsimsel impütasyon ─────────────────────────────────────────────────
dol_monthly_avg = df_dol.copy()
dol_monthly_avg['month'] = dol_monthly_avg.index.month
monthly_avg = dol_monthly_avg.groupby('month')['doluluk'].mean()

# ✅ DÜZELTİLDİ: Sabit '2026-03-31' yerine daily'nin gerçek bitiş tarihi kullanılıyor.
# Böylece daily'deki tüm günler (2026-04-13'e kadar) NaN almadan doluluk alır.
# FORECAST_END sabit kalabilir — bu değişiklik sadece dol_full'un kapsama alanını genişletir.
dol_end   = max(daily['date'].max(), pd.Timestamp(FORECAST_END))
all_dates = pd.date_range(daily['date'].min(), dol_end, freq='D')
dol_full  = pd.Series(index=all_dates, dtype=float, name='doluluk')
dol_full.update(df_dol['doluluk'])
for dt in dol_full[dol_full.isna()].index:
    dol_full[dt] = monthly_avg.get(dt.month, monthly_avg.mean())

dol_roll7 = dol_full.rolling(7, min_periods=1, center=True).mean()
dol_delta = dol_full.diff().fillna(0)

daily['doluluk']   = daily['date'].map(dol_full).values
daily['dol_roll7'] = daily['date'].map(dol_roll7).values
daily['dol_delta'] = daily['date'].map(dol_delta).values

# NaN kontrolü — hata ayıklama
nan_count = daily[['doluluk','dol_roll7','dol_delta']].isna().sum().sum()
print(f'\n✔ Doluluk özellikleri eklendi')
print(f'  dol_full kapsamı  : {all_dates.min().date()} → {all_dates.max().date()}')
print(f'  Kalan NaN sayısı  : {nan_count}  (0 olmalı — sıfır değilse Cell 2\'deki FORECAST_END\'i kontrol et)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Türk Tatil Günleri (2019–2026)                                ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   İki tatil kümesi oluşturur ve birleştirir:
#
#   TR_HOL    — Sabit tarihli resmi ulusal tatiller (her yıl aynı gün):
#               1 Ocak, 23 Nisan, 1 Mayıs, 19 Mayıs, 15 Temmuz,
#               30 Ağustos, 29 Ekim
#
#   RELIGIOUS — Hicri takvime göre kayan dini tatiller:
#               Ramazan Bayramı (3 gün) + Kurban Bayramı (3 gün)
#               Her yıl manuel girilmiştir — 2027 için güncellemeyi unutma.
#
#   ALL_HOL   — TR_HOL | RELIGIOUS — Cell 6 ve Cell 10'da kullanılır.
#
# Genişletme önerileri:
#   • AH'a özgü idari kapatma günleri varsa TR_HOL'a ekle.
#   • Köprü tatilleri AH vaka verisinde belirgin düşüş yaratıyorsa ekle.
#
TR_HOL = set()
for y in range(2019, 2027):
    for m, d in [(1,1),(4,23),(5,1),(5,19),(7,15),(8,30),(10,29)]:
        TR_HOL.add(pd.Timestamp(f'{y}-{m:02d}-{d:02d}'))

RELIGIOUS = {
    # Ramazan Bayramı (Eid al-Fitr)
    pd.Timestamp('2020-05-24'), pd.Timestamp('2020-05-25'), pd.Timestamp('2020-05-26'),
    pd.Timestamp('2021-05-13'), pd.Timestamp('2021-05-14'), pd.Timestamp('2021-05-15'),
    pd.Timestamp('2022-05-02'), pd.Timestamp('2022-05-03'), pd.Timestamp('2022-05-04'),
    pd.Timestamp('2023-04-21'), pd.Timestamp('2023-04-22'), pd.Timestamp('2023-04-23'),
    pd.Timestamp('2024-04-10'), pd.Timestamp('2024-04-11'), pd.Timestamp('2024-04-12'),
    pd.Timestamp('2025-03-30'), pd.Timestamp('2025-03-31'), pd.Timestamp('2025-04-01'),
    pd.Timestamp('2026-03-20'), pd.Timestamp('2026-03-21'), pd.Timestamp('2026-03-22'),
    # Kurban Bayramı (Eid al-Adha)
    pd.Timestamp('2020-07-31'), pd.Timestamp('2020-08-01'), pd.Timestamp('2020-08-02'),
    pd.Timestamp('2021-07-20'), pd.Timestamp('2021-07-21'), pd.Timestamp('2021-07-22'),
    pd.Timestamp('2022-07-09'), pd.Timestamp('2022-07-10'), pd.Timestamp('2022-07-11'),
    pd.Timestamp('2023-06-28'), pd.Timestamp('2023-06-29'), pd.Timestamp('2023-06-30'),
    pd.Timestamp('2024-06-17'), pd.Timestamp('2024-06-18'), pd.Timestamp('2024-06-19'),
    pd.Timestamp('2025-06-06'), pd.Timestamp('2025-06-07'), pd.Timestamp('2025-06-08'),
}
ALL_HOL = TR_HOL | RELIGIOUS
print(f'✔ {len(ALL_HOL)} tatil günü kayıt edildi (2019–2026, ulusal + dini)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Özellik Mühendisliği                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Her gün için modellerin göreceği tüm girdi değişkenlerini üretir.
#   Üretilen özellikler:
#
#   [Zaman]    day_of_year, week_of_year, year
#   [Döngüsel] sin/cos kodlama: dayofweek(7), month(12), quarter(4)
#   [Tatil]    is_holiday, is_weekend
#              near_hol_{-HOL_WINDOW..+HOL_WINDOW} — tatil yakınlık bayrakları
#   [Klinik]   num_prov — o aydaki aktif cerrah sayısı
#   [Pandemi]  is_covid — COVID_END tarihine kadar 1, sonrası 0
#   [Anomali]  is_anomaly — IsolationForest ile işaretlenmiş aykırı günler
#   [Gecikmeli] lag_3/7/14/28/60/90
#              same_week_ly — geçen yıl aynı hafta (pandemi-farkındalıklı)
#   [Rolling]  roll_7/14/28
#   [Doluluk]  dol_ewma — DOL_EWMA_SPAN parametresiyle üstel ağırlıklı ortalama
#
# ⚠️  Parametre notları:
#   contamination=0.05 → günlerin %5'i anomali işaretlenir.
#   [TEST ET] AH'ın pandemi dönemi yoğunluğuna göre 0.03 ile dene.
#
#   AH verisi 2026-04-13'e kadar olduğundan same_week_ly için lag_1yr (364 gün)
#   gerçek 2025 verisine denk gelir — pandemi fallback mekanizması yine de aktif.
#
def engineer_features(df):
    df = df.copy()
    d  = pd.to_datetime(df['date'])

    # Zaman bileşenleri
    df['day_of_year']  = d.dt.dayofyear
    df['week_of_year'] = d.dt.isocalendar().week.astype(int)
    df['year']         = d.dt.year

    # Döngüsel kodlama — periyodik mesafe korunur
    for feat, period in [('dayofweek', 7), ('month', 12), ('quarter', 4)]:
        v = getattr(d.dt, feat)
        df[f'sin_{feat}'] = np.sin(2 * np.pi * v / period)
        df[f'cos_{feat}'] = np.cos(2 * np.pi * v / period)

    # Tatil & hafta sonu bayrakları
    df['is_holiday'] = d.isin(ALL_HOL).astype(int)
    df['is_weekend']  = (d.dt.dayofweek >= 5).astype(int)

    # HOL_WINDOW parametresiyle tatil yakınlık bayrakları
    for off in range(-HOL_WINDOW, HOL_WINDOW + 1):
        if off == 0: continue
        shifted = {h + pd.Timedelta(days=off) for h in ALL_HOL}
        df[f'near_hol_{off:+d}'] = d.isin(shifted).astype(int)

    # Aylık doktor sayısı — ameliyat kapasitesinin proxy'si
    mk = d.dt.to_period('M')
    pm = prov_monthly.set_index('month_key')['num_prov']
    df['num_prov'] = mk.map(pm).ffill().bfill()

    # COVID_END parametresiyle pandemi bayrağı
    df['is_covid'] = ((d >= '2020-01-01') & (d < COVID_END)).astype(int)

    # Anomali bayrağı — IsolationForest
    # [TEST ET] contamination=0.05. AH pandemi yoğunluğuna göre 0.03 ile dene.
    iso = IsolationForest(contamination=0.05, random_state=SEED)
    df['is_anomaly'] = (iso.fit_predict(df[['volume']]) == -1).astype(int)

    # Gecikmeli özellikler
    for lag in [3, 7, 14, 28, 60, 90]:
        df[f'lag_{lag}'] = df['volume'].shift(lag).fillna(0)

    # Geçen yıl aynı hafta (pandemi-farkındalıklı)
    lag_1yr = df['volume'].shift(364)
    lag_2yr = df['volume'].shift(364 * 2)
    lag_3yr = df['volume'].shift(364 * 3)
    df['same_week_ly'] = lag_1yr.fillna(lag_2yr).fillna(lag_3yr).fillna(0)
    covid_mask     = (df['date'] >= '2022-01-01') & (df['date'] < '2024-01-01')
    covid_prev_low = df['same_week_ly'] < 20
    df.loc[covid_mask & covid_prev_low, 'same_week_ly'] = lag_2yr[covid_mask & covid_prev_low].fillna(0)

    # Rolling ortalama
    for win in [7, 14, 28]:
        df[f'roll_{win}'] = (df['volume'].shift(1)
                                        .rolling(win, min_periods=1)
                                        .mean().fillna(0))

    # EWMA doluluk — [TEST ET] span=3 hızlı tepki, span=14 düzgün ama gecikmeli
    df['dol_ewma'] = (df['doluluk'].ewm(span=DOL_EWMA_SPAN, adjust=False).mean())

    return df

daily = engineer_features(daily)

# FEATURE_COLS: HOL_WINDOW'a göre dinamik tatil sütunları
hol_cols = [f'near_hol_{off:+d}'
            for off in range(-HOL_WINDOW, HOL_WINDOW + 1) if off != 0]

FEATURE_COLS = (
    ['volume',
     'day_of_year', 'week_of_year',
     'sin_dayofweek', 'cos_dayofweek',
     'sin_month',    'cos_month',
     'sin_quarter',  'cos_quarter',
     'is_holiday', 'is_weekend']
    + hol_cols
    + ['num_prov', 'is_covid', 'is_anomaly',
       'lag_3', 'lag_7', 'lag_14', 'lag_28', 'lag_60', 'lag_90',
       'same_week_ly',
       'roll_7', 'roll_14', 'roll_28',
       'doluluk', 'dol_roll7', 'dol_delta', 'dol_ewma']
)

print(f'✔ Özellik mühendisliği tamamlandı')
print(f'  Toplam özellik : {len(FEATURE_COLS)}')
print(f'  Tatil yakınlık : {len(hol_cols)} sütun (HOL_WINDOW={HOL_WINDOW})')
print(f'  Anomali sayısı : {daily["is_anomaly"].sum()} gün ({daily["is_anomaly"].mean()*100:.1f}%)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Eğitim/Validasyon Bölünmesi & Ölçekleme                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#
# 1. Zaman serisi bölünmesi:
#    train_df → 2020–2024 (TRAIN_END öncesi)
#    val_df   → 2025 (TRAIN_END ile VAL_END arası)
#    Karıştırma yapılmaz — zaman serisi sırası korunur.
#
# 2. RobustScaler:
#    Her özelliği medyan ve IQR'a göre ölçekler: (x - median) / IQR
#    Pandemi dönemindeki sıfıra yakın günler ölçeklemeyi bozmaz.
#    Scaler SADECE eğitim verisine fit edilir (data leakage önlenir).
#
# 3. Dizi üretimi (make_sequences):
#    LOOKBACK=28, HORIZON=7 → her dizi 28 günü girdi, 7 günü hedef olarak alır.
#
# 4. inv_vol fonksiyonu:
#    Ölçeklenmiş tahmin çıktısını orijinal vaka sayısına geri çevirir.
#    ⚠️  volume sütununun FEATURE_COLS'ta ilk sırada (indeks 0) kalması zorunlu.
#
# ℹ️  AH notu: daily DataFrame 2026-04-13'e kadar gerçek veri içeriyor.
#    TRAIN_END ve VAL_END dışındaki günler (2026 Oca-Mar) all_sc üzerinden
#    tahmin döneminde gerçek veri olarak karşılaştırmaya katılır (has_actual=True).
#
# [TEST ET] VAL_SPLIT = 0.15 → eğitim setinin %15'i iç validasyon için.
#    Dene: 0.10 → 0.15 → 0.20
#
train_df = daily[daily['date'] <  TRAIN_END].copy()
val_df   = daily[(daily['date'] >= TRAIN_END) &
                 (daily['date'] <  VAL_END)].copy()

scaler   = RobustScaler()
train_sc = scaler.fit_transform(train_df[FEATURE_COLS].values)
val_sc   = scaler.transform(val_df[FEATURE_COLS].values)
all_sc   = scaler.transform(daily[FEATURE_COLS].values)

def inv_vol(sv):
    """Ölçeklenmiş hacim değerlerini orijinal ölçeğe geri çevir.
    volume FEATURE_COLS'un ilk elemanı (indeks 0) olmalı.
    """
    d = np.zeros((len(sv), len(FEATURE_COLS)))
    d[:, 0] = sv
    return scaler.inverse_transform(d)[:, 0]

def make_sequences(arr, lb=LOOKBACK, hz=HORIZON):
    """(LOOKBACK, n_features) → HORIZON boyutunda dizi çiftleri üretir."""
    X, y = [], []
    for i in range(len(arr) - lb - hz + 1):
        X.append(arr[i : i + lb])
        y.append(arr[i + lb : i + lb + hz, 0])
    return np.array(X), np.array(y)

X_tr, y_tr = make_sequences(train_sc)
X_va, y_va = make_sequences(np.concatenate([train_sc[-LOOKBACK:], val_sc]))

print(f'Eğitim : {len(train_df):,} gün  ({train_df["date"].min().date()} – {train_df["date"].max().date()})')
print(f'Val    : {len(val_df):,} gün   ({val_df["date"].min().date()} – {val_df["date"].max().date()})')
print(f'LSTM dizisi — Eğitim: {len(X_tr):,}  |  Val: {len(X_va):,}')
print(f'Dizi şekli: {X_tr.shape}  →  Hedef şekli: {y_tr.shape}')
print(f'Scaler   : RobustScaler  (center={scaler.center_[0]:.2f}  scale={scaler.scale_[0]:.2f})')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — MODEL 1: Encoder-Decoder seq2seq LSTM                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#
# Mimari (build_lstm fonksiyonu):
#   Encoder:
#     • Bidirectional LSTM — çift yönlü örüntü öğrenimi
#     • Standart LSTM — son gizli durum [h, c]'yi decoder'a aktarır
#
#   Decoder:
#     • RepeatVector: encoder çıktısını HORIZON kez tekrarlar
#     • Decoder LSTM-1 (LSTM_UNITS): encoder bağlamını alır
#     • Decoder LSTM-2 (LSTM_UNITS//2): daralan huni yapısı
#       [TEST ET] LSTM_UNITS//4 ile daha dar şişe boynu dene
#     • TimeDistributed(Dense(1)): her adım için tek tahmin
#
# Kayıp fonksiyonu:
#   Huber(delta=HUBER_DELTA) — pandemi aykırı günlerine dayanıklı.
#
# Callbacks:
#   EarlyStopping  → val_loss EARLY_PATIENCE epoch iyileşmezse durur
#   ReduceLROnPlateau → REDUCE_PATIENCE epoch iyileşmezse LR'yi REDUCE_FACTOR ile çarpar
#
# [TEST ET] REDUCE_FACTOR=0.3 daha agresif LR düşürümü sağlar.
#
def build_lstm(n_features):
    enc_in = Input(shape=(LOOKBACK, n_features), name='encoder_input')

    # Bidirectional LSTM — çift yönlü örüntü öğrenimi
    x      = Bidirectional(
                 LSTM(LSTM_UNITS, return_sequences=True,
                      dropout=DROPOUT,
                      recurrent_dropout=RECURRENT_DROPOUT),
                 name='bi_lstm'
             )(enc_in)
    _, h, c = LSTM(LSTM_UNITS, return_state=True,
                   dropout=DROPOUT,
                   recurrent_dropout=RECURRENT_DROPOUT,
                   name='encoder_lstm')(x)

    # Decoder — 2 katmanlı daralan yapı
    dec  = RepeatVector(HORIZON, name='repeat')(h)
    dec  = LSTM(LSTM_UNITS, return_sequences=True,
                dropout=DROPOUT, recurrent_dropout=RECURRENT_DROPOUT,
                name='decoder_lstm_1')(dec, initial_state=[h, c])
    dec  = LSTM(LSTM_UNITS // 2, return_sequences=True,
                dropout=DROPOUT,
                name='decoder_lstm_2')(dec)
    out  = TimeDistributed(Dense(1), name='output')(dec)
    out  = tf.keras.layers.Reshape((HORIZON,))(out)

    model = Model(enc_in, out, name='seq2seq_bilstm_v1_ah')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,
            clipnorm=CLIP_NORM
        ),
        loss=tf.keras.losses.Huber(delta=HUBER_DELTA)
    )
    return model

lstm_model = build_lstm(len(FEATURE_COLS))
lstm_model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=EARLY_PATIENCE,
                  restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=REDUCE_FACTOR,
                      patience=REDUCE_PATIENCE, min_lr=1e-7, verbose=1),
]

print(f'\n⏳ LSTM eğitiliyor …')
print(f'   LR={LEARNING_RATE}  clipnorm={CLIP_NORM}  Huber_δ={HUBER_DELTA}')
print(f'   recurrent_dropout={RECURRENT_DROPOUT}  patience={EARLY_PATIENCE}')
print(f'   (GPU ile ~5-15 dk, CPU ile ~50-70 dk)')
history = lstm_model.fit(
    X_tr, y_tr,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=callbacks,
    verbose=1
)
print(f'✔ LSTM eğitimi tamamlandı ({len(history.history["loss"])} epoch)')

# Kayıp eğrisi — val ve eğitim kaybı çok ayrışıyorsa DROPOUT artır
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     color='#3B82F6', lw=1.5, label='Eğitim Kaybı')
ax.plot(history.history['val_loss'], color='#F59E0B', lw=1.5, ls='--', label='Val Kaybı')
ax.set_title(f'LSTM — Eğitim & Validasyon Kaybı (Huber δ={HUBER_DELTA})', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Kayıp')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — LSTM Tahmin Fonksiyonu (Direct Forecast)                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   direct_forecast_lstm fonksiyonu HORIZON=7 günlük bloklarla tahmin üretir.
#
# Neden 'direct' yöntem:
#   Her blok için son LOOKBACK günün GERÇEK verisi kullanılır → hata birikimi yok.
#
# ℹ️  AH notu: Tahmin modu (mode='test') all_sc'nin son LOOKBACK gününü kullanır.
#   AH verisi 2026-04-13'e kadar gerçek veri içerdiğinden 2026 Oca-Mar tahmini
#   2025 yılsonuna dayalı gerçek pencereden üretilir — yüksek kaliteli başlangıç noktası.
#
def direct_forecast_lstm(mode='val'):
    if mode == 'val':
        target_dates = val_df['date'].values
        n_target     = len(val_df)
    else:
        target_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')
        n_target     = len(target_dates)

    results = []
    dates   = pd.DatetimeIndex(pd.to_datetime(list(target_dates)))

    for i in range(0, n_target, HORIZON):
        chunk = dates[i : i + HORIZON]
        if mode == 'val':
            real_past = np.vstack([train_sc, val_sc[:i]])
        else:
            real_past = all_sc
        seed  = real_past[-LOOKBACK:]
        x_in  = seed.reshape(1, LOOKBACK, len(FEATURE_COLS))
        ps    = lstm_model.predict(x_in, verbose=0)[0]
        pv    = np.clip(inv_vol(ps), 0, None)
        for dt, p in zip(chunk, pv):
            results.append({'date': pd.Timestamp(dt), 'pred_lstm': round(float(p))})

    return pd.DataFrame(results)

fcast_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')

print('⏳ LSTM validasyon tahmini …')
lstm_val  = direct_forecast_lstm(mode='val')
print('⏳ LSTM 2026 tahmini …')
lstm_test = direct_forecast_lstm(mode='test')
print(f'✔ LSTM: {len(lstm_val)} validasyon, {len(lstm_test)} tahmin')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — MODEL 2: Facebook Prophet                                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Prophet trend + mevsimsellik + tatil + dışsal regressor bileşenleriyle
#   ayrıştırılmış bir zaman serisi modeli kurar.
#
# Bileşenler:
#   • changepoint_prior_scale  → trendin ne kadar esnek değiştiği
#   • seasonality_mode=multiplicative → oransal mevsimsel etki
#   • yearly_seasonality=PROPHET_YEARLY_N → Fourier bileşeni sayısı
#   • Regressorlar: num_prov, is_covid, doluluk
#
# ⚠️  AH notu:
#   • is_covid 2026 tahmin döneminde sıfır geçilir — doğru.
#   • doluluk AH'ta Kas 2025'te bitiyor. 2026 için mevsimsel ortalamaya fallback.
#     Gerçek 2026 doluluk verisi elde edilirse dol_full serisini güncelle.
#
# [TEST ET] PROPHET_CP_SCALE — Bu modelin en kritik parametresi.
#   Bileşen grafiğinde trend düz/garip görünüyorsa:
#     Çok düz → 0.3 veya 0.5'e çık
#     Çok kıvrımlı / overfit → 0.05'e düşür
#
print('⏳ Prophet eğitiliyor …')

hol_df = pd.DataFrame([
    {'holiday': 'tr_tatil', 'ds': h,
     'lower_window': -2, 'upper_window': 2}
    for h in ALL_HOL
])

def get_prov(dt):
    mk = pd.Period(pd.Timestamp(dt), 'M')
    pm = prov_monthly.set_index('month_key')['num_prov']
    return float(pm.get(mk, pm.iloc[-1]))

def get_doluluk(dt):
    """Tarih için doluluk oranı döndür (impüte edilmiş dol_full serisinden)."""
    ts = pd.Timestamp(dt).normalize()
    return float(dol_full.get(ts, dol_full.mean()))

prophet_model = Prophet(
    seasonality_mode=PROPHET_SEAS_MODE,
    weekly_seasonality=True,
    yearly_seasonality=PROPHET_YEARLY_N,
    daily_seasonality=False,
    holidays=hol_df,
    changepoint_prior_scale=PROPHET_CP_SCALE,   # [TEST ET] en kritik parametre
    seasonality_prior_scale=PROPHET_SEAS_PRIOR,
    holidays_prior_scale=PROPHET_HOL_PRIOR,
)
prophet_model.add_regressor('num_prov', standardize=True)
prophet_model.add_regressor('is_covid', standardize=False)
prophet_model.add_regressor('doluluk',  standardize=True)

train_prophet = train_df[['date','volume','is_covid']].rename(
    columns={'date':'ds','volume':'y'}).copy()
train_prophet['num_prov'] = train_prophet['ds'].apply(get_prov)
train_prophet['doluluk']  = train_prophet['ds'].apply(get_doluluk)
prophet_model.fit(train_prophet)

n_future = len(val_df) + len(fcast_dates)
future   = prophet_model.make_future_dataframe(periods=n_future, freq='D')
future['num_prov'] = future['ds'].apply(get_prov)
future['is_covid'] = ((future['ds'] >= '2020-01-01') &
                      (future['ds'] <  COVID_END)).astype(int)
future['doluluk']  = future['ds'].apply(get_doluluk)
fc_prophet = prophet_model.predict(future)

prop_val  = (fc_prophet[fc_prophet['ds'].isin(val_df['date'])]
             [['ds','yhat']]
             .rename(columns={'ds':'date','yhat':'pred_prophet'}))
prop_val['pred_prophet'] = np.clip(prop_val['pred_prophet'], 0, None).round()

prop_test = (fc_prophet[fc_prophet['ds'].isin(fcast_dates)]
             [['ds','yhat']]
             .rename(columns={'ds':'date','yhat':'pred_prophet'}))
prop_test['pred_prophet'] = np.clip(prop_test['pred_prophet'], 0, None).round()

print('✔ Prophet tamamlandı')
print('ℹ️  Bileşen grafiği: trend düz/garip görünüyorsa PROPHET_CP_SCALE değerini ayarla')

# Bileşen grafiği — trend, haftalık mevsimsellik, yıllık mevsimsellik, tatil etkisi
fig = prophet_model.plot_components(fc_prophet)
fig.suptitle('Prophet — Trend & Mevsimsellik Bileşenleri — AH (2020–2026)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — MODEL 3: LightGBM                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   HORIZON=7 adımın her biri için bağımsız bir LGBMRegressor eğitir.
#   (7 model × LGB_N_ESTIMATORS ağaç)
#
# Neden her adım için ayrı model:
#   Her adımın farklı örüntülerini bağımsız öğrenmesine izin verir.
#
# ⚠️  Early stopping henüz eklenmedi. Eklemek için:
#
#   # Cell 7'nin sonuna ekle:
#   X_va_f, y_va_f = make_seq_flat(val_sc)
#
#   # m.fit(...) satırını şununla değiştir:
#   m.fit(X_tr_f, y_tr_f[:, step],
#         eval_set=[(X_va_f, y_va_f[:, step])],
#         callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
#
#   Early stopping eklenirse LGB_N_ESTIMATORS 500-1000'e güvenle çıkabilir.
#
print('⏳ LightGBM eğitiliyor …')

def make_seq_flat(arr, lb=LOOKBACK, hz=HORIZON):
    """LSTM dizilerini LightGBM için (LOOKBACK*n_features,) düz vektöre çevirir."""
    X, y = [], []
    for i in range(len(arr) - lb - hz + 1):
        X.append(arr[i : i + lb].flatten())
        y.append(arr[i + lb : i + lb + hz, 0])
    return np.array(X), np.array(y)

X_tr_f, y_tr_f = make_seq_flat(train_sc)

lgb_models = []
for step in range(HORIZON):
    m = lgb.LGBMRegressor(
        n_estimators     = LGB_N_ESTIMATORS,  # [TEST ET] early stopping ile 500+ dene
        max_depth        = LGB_MAX_DEPTH,     # [TEST ET] 4 daha basit, 8 overfitting riski
        learning_rate    = LGB_LR,
        subsample        = LGB_SUBSAMPLE,
        subsample_freq   = 1,
        colsample_bytree = LGB_COLSAMPLE,
        min_child_samples= LGB_MIN_LEAF,
        random_state     = SEED,
        n_jobs           = -1,
        verbose          = -1
    )
    m.fit(X_tr_f, y_tr_f[:, step])
    lgb_models.append(m)

print(f'✔ LightGBM tamamlandı ({HORIZON} model × {LGB_N_ESTIMATORS} ağaç)')
print(f'   depth={LGB_MAX_DEPTH}  lr={LGB_LR}  subsample={LGB_SUBSAMPLE}  col={LGB_COLSAMPLE}')

def direct_forecast_lgb(mode='val'):
    """LSTM ile aynı direct forecast mantığı — hata birikimi yok."""
    if mode == 'val':
        target_dates = val_df['date'].values
        n_target     = len(val_df)
    else:
        target_dates = fcast_dates
        n_target     = len(fcast_dates)

    results = []
    dates   = pd.DatetimeIndex(pd.to_datetime(list(target_dates)))

    for i in range(0, n_target, HORIZON):
        chunk = dates[i : i + HORIZON]
        if mode == 'val':
            real_past = np.vstack([train_sc, val_sc[:i]])
        else:
            real_past = all_sc
        seed = real_past[-LOOKBACK:]
        flat = seed.flatten().reshape(1, -1)
        ps   = np.array([m.predict(flat)[0] for m in lgb_models])
        pv   = np.clip(inv_vol(ps), 0, None)
        for dt, p in zip(chunk, pv):
            results.append({'date': pd.Timestamp(dt), 'pred_lgb': round(float(p))})

    return pd.DataFrame(results)

gbm_val  = direct_forecast_lgb(mode='val')
gbm_test = direct_forecast_lgb(mode='test')
gbm_val['pred_gbm']  = gbm_val['pred_lgb']
gbm_test['pred_gbm'] = gbm_test['pred_lgb']
print(f'✔ LightGBM: {len(gbm_val)} validasyon, {len(gbm_test)} tahmin')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Metrikler, Karşılaştırma & Ağırlıklı Ensemble               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#
# 1. compute_metrics: R², RMSE, MAE hesaplar.
#    Yalnızca volume>0 olan günler değerlendirmeye alınır.
#
# 2. Metrik tablosu: üç modeli referans paper sonucuyla karşılaştırır.
#    Durum eşikleri: R²≥0.7 → İyi, ≥0.4 → Orta, <0.4 → Düşük
#
# 3. Ağırlıklı Ensemble:
#    Her modelin val R²'si ağırlık olarak kullanılır.
#    Negatif R² → 0 ağırlık (modeli ensemble'dan çıkarır).
#
# ℹ️  AH notu: AH verisi 2026 Oca-Mar için gerçek veri içerdiğinden
#    test_comp['volume'].sum() > 0 → has_actual=True → Cell 13'te 2026 grafiğinde
#    gerçek metrikler (R², MAE) de hesaplanıp gösterilir.
#
# [TEST ET] Alternatif ağırlıklandırma — 1/RMSE (daha agresif cezalandırma):
#    rmse_lstm = compute_metrics(val_comp['volume'], val_comp['pred_lstm'])[1]
#    rmse_prop = compute_metrics(val_comp['volume'], val_comp['pred_prophet'])[1]
#    rmse_gbm  = compute_metrics(val_comp['volume'], val_comp['pred_gbm'])[1]
#    w_raw = np.array([1/rmse_lstm, 1/rmse_prop, 1/rmse_gbm])
#
def compute_metrics(y_true, y_pred):
    yt = np.array(y_true); yp = np.array(y_pred)
    m  = yt > 0
    return (r2_score(yt[m], yp[m]),
            np.sqrt(mean_squared_error(yt[m], yp[m])),
            mean_absolute_error(yt[m], yp[m]))

def build_comparison(actual_df, *pred_dfs):
    merged = actual_df[['date','volume']].copy()
    for pdf in pred_dfs:
        merged = merged.merge(pdf, on='date', how='left')
    return merged.fillna(0)

val_comp = build_comparison(val_df, lstm_val, prop_val, gbm_val)

# AH: 2026 Oca-Mar gerçek veri daily'de mevcut → has_actual=True
actual_2026 = daily[daily['date'] >= pd.Timestamp(FORECAST_START)][['date','volume']]

test_comp = pd.DataFrame({'date': fcast_dates})
for pdf in [lstm_test, prop_test, gbm_test]:
    test_comp = test_comp.merge(pdf, on='date', how='left')
test_comp = test_comp.merge(actual_2026, on='date', how='left')
test_comp = test_comp.fillna(0)
test_comp['is_weekend'] = pd.DatetimeIndex(test_comp['date']).dayofweek >= 5
test_comp['is_holiday'] = pd.DatetimeIndex(test_comp['date']).isin(ALL_HOL)

print('='*62)
print(f"  Validasyon Performansı — 2025  (AH)")
print(f"  {'Model':<16} {'R²':>6} {'RMSE':>7} {'MAE':>7}  {'Durum':<12}")
print('  ' + '─'*56)
for col, name in [('pred_lstm','LSTM'),('pred_prophet','Prophet'),('pred_gbm','LightGBM')]:
    r2, rm, ma = compute_metrics(val_comp['volume'], val_comp[col])
    durum = '✅ İyi' if r2 >= 0.7 else ('⚠️ Orta' if r2 >= 0.4 else '❌ Düşük')
    print(f"  {name:<16} {r2:>6.3f} {rm:>7.2f} {ma:>7.2f}  {durum}")
print(f"  {'Paper LSTM':<16} {'0.855':>6} {'2.017':>7} {'1.104':>7}  (referans)")
print('='*62)

print()
print('='*62)
print('  Ağırlıklı Ensemble (val R²)')

R2_lstm    = compute_metrics(val_comp['volume'], val_comp['pred_lstm'])[0]
R2_prophet = compute_metrics(val_comp['volume'], val_comp['pred_prophet'])[0]
R2_gbm     = compute_metrics(val_comp['volume'], val_comp['pred_gbm'])[0]

w_raw = np.array([max(R2_lstm, 0), max(R2_prophet, 0), max(R2_gbm, 0)])
if w_raw.sum() == 0:
    w_raw = np.ones(3)
weights   = w_raw / w_raw.sum()
w_lstm, w_prophet, w_gbm = weights

print(f'  Ağırlıklar: LSTM={w_lstm:.3f}  Prophet={w_prophet:.3f}  LightGBM={w_gbm:.3f}')

val_comp['pred_ensemble'] = (
    w_lstm    * val_comp['pred_lstm'] +
    w_prophet * val_comp['pred_prophet'] +
    w_gbm     * val_comp['pred_gbm']
).round()
test_comp['pred_ensemble'] = (
    w_lstm    * test_comp['pred_lstm'] +
    w_prophet * test_comp['pred_prophet'] +
    w_gbm     * test_comp['pred_gbm']
).round()

r2e, rme, mae_e = compute_metrics(val_comp['volume'], val_comp['pred_ensemble'])
durum_e = '✅ İyi' if r2e >= 0.7 else ('⚠️ Orta' if r2e >= 0.4 else '❌ Düşük')
print(f'  Ensemble         {r2e:>6.3f} {rme:>7.2f} {mae_e:>7.2f}  {durum_e}')
print('='*62)

# 2026 gerçek vs tahmin metrikleri (has_actual=True ise)
has_actual = test_comp['volume'].sum() > 0
if has_actual:
    print()
    print('='*62)
    print('  2026 Oca-Mar Gerçek vs Tahmin (AH — has_actual=True)')
    print(f"  {'Model':<16} {'R²':>6} {'RMSE':>7} {'MAE':>7}")
    print('  ' + '─'*56)
    for col, name in [('pred_lstm','LSTM'),('pred_prophet','Prophet'),
                      ('pred_gbm','LightGBM'),('pred_ensemble','Ensemble')]:
        r2, rm, ma = compute_metrics(test_comp['volume'], test_comp[col])
        print(f"  {name:<16} {r2:>6.3f} {rm:>7.2f} {ma:>7.2f}")
    print('='*62)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Ana Görselleştirme (6 Satır × 3 Sütun GridSpec)             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Tüm modellerin performansını ve 2026 tahminlerini tek büyük figürde gösterir.
#
# Satır düzeni:
#   Satır 0 — 2025 Validasyon: Gerçek vs tüm modeller + ensemble
#   Satır 1 — 2026 Tahmin: hafta sonu/tatil gri bantlarıyla
#              ℹ️  AH'ta has_actual=True → 2026 Oca-Mar gerçek veri bar grafik olarak görünür
#   Satır 2 — Scatter: her model için tahmin vs gerçek (validasyon)
#
COLS = {'LSTM':'#3B82F6', 'Prophet':'#F59E0B', 'GBM':'#10B981', 'Ensemble':'#8B5CF6', 'Actual':'#1E293B'}
has_actual = test_comp['volume'].sum() > 0

fig = plt.figure(figsize=(18, 34), facecolor='#F8FAFC')
gs  = gridspec.GridSpec(6, 3, figure=fig, hspace=0.52, wspace=0.32,
                        top=0.95, bottom=0.03, left=0.06, right=0.97)

# ── Satır 0: Validasyon 2025 ──────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0, :]); ax0.set_facecolor('white')
ax0.plot(val_df['date'], val_df['volume'],
         color=COLS['Actual'], lw=2, label='Gerçek', zorder=5)
for col, name, ls in [('pred_lstm','LSTM','--'),
                       ('pred_prophet','Prophet',':'),
                       ('pred_gbm','LightGBM','-.'),
                       ('pred_ensemble','Ensemble','-')]:
    r2, rm, ma = compute_metrics(val_comp['volume'], val_comp[col])
    ax0.plot(val_df['date'], val_comp[col], color=COLS.get(name, COLS['GBM']), lw=1.7, ls=ls.strip(),
             label=f'{name:<12} R²={r2:.3f}  RMSE={rm:.1f}  MAE={ma:.1f}')
ax0.set_title('Validasyon — 2025: Tüm Modeller + Ensemble vs Gerçek Günlük Vaka  (AH)',
              fontsize=12, fontweight='bold')
ax0.set_ylabel('Günlük Vaka')
ax0.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax0.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax0.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax0.legend(fontsize=9, ncol=2, loc='upper left', prop={'family':'monospace'})
ax0.grid(alpha=0.2, lw=0.5)

# ── Satır 1: Ocak-Mart 2026 Tahminleri ───────────────────────────────────────
# AH'ta has_actual=True — gerçek veri bar grafik olarak gösterilir,
# tahminler üzerine çizilir → gerçek model karşılaştırması mümkün
ax1 = fig.add_subplot(gs[1, :]); ax1.set_facecolor('white')
for _, r in test_comp[test_comp['is_weekend'] | test_comp['is_holiday']].iterrows():
    ax1.axvspan(r['date'] - pd.Timedelta(hours=12),
                r['date'] + pd.Timedelta(hours=12),
                alpha=0.07, color='gray')
if has_actual:
    ax1.bar(test_comp['date'], test_comp['volume'],
            color='#CBD5E1', width=0.85, label='Gerçek', zorder=2)
for col, name, ls in [('pred_lstm','LSTM','--'),
                       ('pred_prophet','Prophet',':'),
                       ('pred_gbm','LightGBM','-.'),
                       ('pred_ensemble','Ensemble','-')]:
    lbl = name
    if has_actual:
        r2, rm, ma = compute_metrics(test_comp['volume'], test_comp[col])
        lbl = f'{name}  R²={r2:.3f}  MAE={ma:.1f}'
    ax1.plot(test_comp['date'], test_comp[col], color=COLS.get(name, COLS['GBM']),
             lw=2.2, ls=ls.strip(), label=lbl, zorder=5)
ax1.set_title('Tahmin — Ocak–Mart 2026 (gri = hafta sonu/tatil)  [AH — Gerçek Veri Dahil]',
              fontsize=12, fontweight='bold')
ax1.set_ylabel('Günlük Vaka')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax1.legend(fontsize=9, ncol=4, prop={'family':'monospace'})
ax1.grid(alpha=0.2, lw=0.5)

# ── Satır 2: Scatter grafikleri (validasyon) ──────────────────────────────────
# İdeal: noktalar y=x çizgisinin üzerinde. Sistematik sapma → model bias'ı.
for ci, (col, name) in enumerate([('pred_lstm','LSTM'),
                                    ('pred_prophet','Prophet'),
                                    ('pred_gbm','LightGBM')]):
    ax = fig.add_subplot(gs[2, ci]); ax.set_facecolor('white')
    mask = val_comp['volume'] > 0
    ax.scatter(val_comp.loc[mask,'volume'], val_comp.loc[mask,col],
               alpha=0.4, s=8, color=COLS.get(name, '#888'))
    mn = min(val_comp.loc[mask,'volume'].min(), val_comp.loc[mask,col].min())
    mx = max(val_comp.loc[mask,'volume'].max(), val_comp.loc[mask,col].max())
    ax.plot([mn,mx],[mn,mx], 'k--', lw=0.8, alpha=0.5)
    r2, rm, ma = compute_metrics(val_comp['volume'], val_comp[col])
    ax.set_title(f'{name}  R²={r2:.3f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Gerçek'); ax.set_ylabel('Tahmin')
    ax.grid(alpha=0.2, lw=0.5)

plt.suptitle('AH Ameliyat Hacmi Tahmin Modeli — v1 Sonuçları', fontsize=14, fontweight='bold', y=0.97)
plt.savefig('ah_forecast_v1.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Grafik kaydedildi: ah_forecast_v1.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Modelleri Kaydet & İndir                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Eğitilmiş modelleri ve tahmin sonuçlarını kaydeder, Colab'dan indirir.
#
# Kaydedilen dosyalar:
#   ah_lstm_v1.keras             — TensorFlow/Keras LSTM modeli
#   ah_lgb_scaler_v1.pkl         — LightGBM modelleri + RobustScaler +
#                                   FEATURE_COLS + ensemble ağırlıkları
#   ah_tahmin_oca_mar_2026.csv   — 2026 Oca-Mar tahminleri + gerçek veri
#   ah_validasyon_2025.csv       — 2025 validasyon sonuçları
#   ah_forecast_v1.png           — ana görselleştirme grafiği
#
# ⚠️  Prophet modeli kaydedilmiyor. Kaydetmek için:
#   from prophet.serialize import model_to_json
#   with open('ah_prophet_v1.json', 'w') as f:
#       f.write(model_to_json(prophet_model))
#   files.download('ah_prophet_v1.json')
#
from google.colab import files
import pickle

lstm_model.save('ah_lstm_v1.keras')

with open('ah_lgb_scaler_v1.pkl', 'wb') as f:
    pickle.dump({'lgb_models': lgb_models, 'scaler': scaler,
                 'feature_cols': FEATURE_COLS,
                 'weights': {'lstm': w_lstm, 'prophet': w_prophet, 'lgb': w_gbm}}, f)

test_comp.to_csv('ah_tahmin_oca_mar_2026.csv', index=False)
val_comp.to_csv('ah_validasyon_2025.csv', index=False)

for fname in ['ah_forecast_v1.png',
              'ah_tahmin_oca_mar_2026.csv',
              'ah_validasyon_2025.csv',
              'ah_lstm_v1.keras']:
    try:
        files.download(fname)
    except Exception as e:
        print(f'  ⚠️ {fname}: {e}')

print('\n✅ Tüm dosyalar indirildi!')
print('  📊 ah_forecast_v1.png                — ana grafik (ensemble + gerçek veri)')
print('  📋 ah_tahmin_oca_mar_2026.csv        — Oca-Mar 2026 tahminleri + gerçek')
print('  📋 ah_validasyon_2025.csv            — 2025 validasyon sonuçları')
print('  🤖 ah_lstm_v1.keras                  — eğitilmiş LSTM modeli')
print('  📦 ah_lgb_scaler_v1.pkl              — LightGBM modelleri + ensemble ağırlıkları')
print()
print('  ℹ️  Prophet modeli kaydedilmedi. Yukarıdaki yorumlara bak.')